# Milestone 2 Live Market Walkthrough

This notebook is a hands-on harness for the current Milestone 2 surface.

It reuses the existing CLI collectors to:
- backfill one active market into isolated per-run storage
- inspect the selected market and token ids from DuckDB
- record a 300-second live market-channel session for every token in that market
- visualize historical prices, live top-of-book state, spreads, and trades
- inspect raw websocket captures and row-count quality checks


## Workflow

Run the notebook top to bottom for a fresh session, or set `POLYMARKET_EXPLORATION_RUN_ID` before opening Jupyter to revisit an existing run directory.

The notebook keeps all artifacts under `data/runs/<run_id>/` so the exploration does not mix with the shared default warehouse.


In [ ]:
%matplotlib inline

import importlib
import json
import os
import shlex
import subprocess
import sys
from collections import Counter, defaultdict
from datetime import UTC, datetime
from decimal import Decimal
from pathlib import Path
from pprint import pprint

import duckdb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from dotenv import load_dotenv

REPO_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if candidate.name == "polymarket-signals-whales":
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root from the current working directory.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.clients.rest import parse_optional_datetime, parse_string_tuple

load_dotenv(REPO_ROOT / ".env")

RUN_ID = os.environ.get(
    "POLYMARKET_EXPLORATION_RUN_ID",
    datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ"),
)
RUN_ROOT = REPO_ROOT / "data" / "runs" / RUN_ID
RAW_DIR = RUN_ROOT / "raw"
WAREHOUSE_PATH = RUN_ROOT / "warehouse" / "polymarket.duckdb"
LIVE_SESSION_SECONDS = 300
BACKFILL_SAMPLE_SIZE = 1

RUN_ROOT.mkdir(parents=True, exist_ok=True)
WAREHOUSE_PATH.parent.mkdir(parents=True, exist_ok=True)

plt.style.use("ggplot")

def run_command(args: list[str]) -> subprocess.CompletedProcess[str]:
    print(f"$ {shlex.join(args)}")
    result = subprocess.run(
        args,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {shlex.join(args)}")
    return result

def fetch_rows(sql: str, params: list[object] | tuple[object, ...] | None = None) -> list[dict[str, object]]:
    params = params or []
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        cursor = connection.execute(sql, params)
        columns = [column[0] for column in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]

def print_rows(rows: list[dict[str, object]], *, limit: int | None = None) -> None:
    if not rows:
        print("<no rows>")
        return
    for row in rows[: limit or len(rows)]:
        pprint(row)

def latest_capture_path(source: str, dataset: str) -> Path | None:
    capture_paths = sorted((RAW_DIR / source / dataset).glob("date=*/*"))
    json_paths = [path for path in capture_paths if path.suffix in {".json", ".jsonl"}]
    return json_paths[-1] if json_paths else None

def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

def parse_sequence(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value if str(item).strip()]
    if isinstance(value, tuple):
        return [str(item) for item in value if str(item).strip()]
    if isinstance(value, str):
        return list(parse_string_tuple(value))
    return []

def extract_market_details_from_gamma_raw(selected_market_id: str) -> dict[str, object]:
    gamma_capture = latest_capture_path("gamma", "sample_market_selection")
    if gamma_capture is None:
        return {"market": None, "labels": {}, "outcome_prices": {}}

    payload = load_json(gamma_capture).get("payload")
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        records = payload.get("data", [])
    else:
        records = []

    for record in records:
        if not isinstance(record, dict):
            continue
        market_id = str(record.get("id") or "").strip()
        if market_id != selected_market_id:
            continue
        token_ids = parse_sequence(record.get("clobTokenIds") or record.get("clob_token_ids"))
        outcomes = parse_sequence(record.get("outcomes"))
        outcome_prices = parse_sequence(record.get("outcomePrices") or record.get("outcome_prices"))
        labels = {
            token_id: outcomes[index] if index < len(outcomes) and outcomes[index] else f"Token {index}"
            for index, token_id in enumerate(token_ids)
        }
        prices = {
            token_id: outcome_prices[index]
            for index, token_id in enumerate(token_ids)
            if index < len(outcome_prices)
        }
        return {"market": record, "labels": labels, "outcome_prices": prices}
    return {"market": None, "labels": {}, "outcome_prices": {}}

def token_label(token_id: str, labels: dict[str, str], token_index_lookup: dict[str, int]) -> str:
    if token_id in labels and labels[token_id]:
        return labels[token_id]
    if token_id in token_index_lookup:
        return f"Token {token_index_lookup[token_id]}"
    return token_id

print(f"REPO_ROOT={REPO_ROOT}")
print(f"RUN_ID={RUN_ID}")
print(f"RAW_DIR={RAW_DIR}")
print(f"WAREHOUSE_PATH={WAREHOUSE_PATH}")


In [ ]:
REQUIRED_MODULES = ("duckdb", "websockets", "matplotlib", "notebook")
for module_name in REQUIRED_MODULES:
    module = importlib.import_module(module_name)
    print(f"{module_name}: {getattr(module, '__version__', 'version unavailable')}")

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")


## 1. Backfill One Active Market

This cell reuses `scripts/backfill_sample_markets.py` with `--sample-size 1` and isolated run paths.


In [ ]:
run_command(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "backfill_sample_markets.py"),
        "--sample-size",
        str(BACKFILL_SAMPLE_SIZE),
        "--raw-dir",
        str(RAW_DIR),
        "--warehouse-path",
        str(WAREHOUSE_PATH),
    ]
)

selected_markets = fetch_rows(
    """
    SELECT market_id, question, condition_id, active, end_time_utc, liquidity, volume, collection_time_utc
    FROM markets
    ORDER BY collection_time_utc DESC, market_id ASC
    """
)
if not selected_markets:
    raise RuntimeError("Backfill completed without writing any market rows.")

selected_market = selected_markets[0]
token_rows = fetch_rows(
    """
    SELECT token_id, token_index
    FROM market_tokens
    WHERE market_id = ?
    ORDER BY token_index ASC
    """,
    [selected_market["market_id"]],
)
if not token_rows:
    raise RuntimeError("No market_tokens were written for the selected market.")

TOKEN_IDS = [str(row["token_id"]) for row in token_rows]
TOKEN_INDEX_LOOKUP = {str(row["token_id"]): int(row["token_index"]) for row in token_rows}
MARKET_CONTEXT = extract_market_details_from_gamma_raw(str(selected_market["market_id"]))
OUTCOME_LABELS = MARKET_CONTEXT["labels"]
OUTCOME_PRICES = MARKET_CONTEXT["outcome_prices"]

print("Selected market:")
pprint(selected_market)
print("\nMarket tokens:")
print_rows(token_rows)
print("\nOutcome labels from raw Gamma capture:")
pprint(OUTCOME_LABELS)
print("\nOutcome prices from raw Gamma capture:")
pprint(OUTCOME_PRICES)


## 2. Record A Live Session

This cell reuses `scripts/record_live_market_stream.py` and subscribes to every token in the selected market for a 300-second session.


In [ ]:
live_command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "record_live_market_stream.py"),
    "--session-seconds",
    str(LIVE_SESSION_SECONDS),
    "--raw-dir",
    str(RAW_DIR),
    "--warehouse-path",
    str(WAREHOUSE_PATH),
]
for token_id in TOKEN_IDS:
    live_command.extend(["--asset-id", token_id])

run_command(live_command)


## 3. Quality Checks And Raw Capture Inspection

Inspect normalized row counts and the most recent websocket raw capture before plotting.


In [ ]:
row_counts = {
    "markets": fetch_rows("SELECT COUNT(*) AS row_count FROM markets")[0]["row_count"],
    "market_tokens": fetch_rows("SELECT COUNT(*) AS row_count FROM market_tokens")[0]["row_count"],
    "price_history": fetch_rows("SELECT COUNT(*) AS row_count FROM price_history")[0]["row_count"],
    "order_book_snapshots": fetch_rows("SELECT COUNT(*) AS row_count FROM order_book_snapshots")[0]["row_count"],
    "trades": fetch_rows("SELECT COUNT(*) AS row_count FROM trades")[0]["row_count"],
}
pprint(row_counts)

websocket_capture = latest_capture_path("websocket", "live_market_channel_events")
if websocket_capture is None:
    print("No websocket raw capture found under data/raw/websocket/live_market_channel_events.")
else:
    websocket_record = load_json(websocket_capture)
    print(f"Latest websocket raw capture: {websocket_capture}")
    pprint(websocket_record["metadata"])
    message = websocket_record["payload"].get("message")
    if isinstance(message, list):
        print("Observed event types:", Counter(item.get("event_type", "<missing>") for item in message if isinstance(item, dict)))
        print("First websocket event preview:")
        pprint(message[0] if message else {})
    else:
        print("Single websocket message payload:")
        pprint(message)


## 4. Market Summary And Historical Price View

Plot the one-week `price_history` backfill for every token in the selected market and label each line with Gamma outcomes when available.


In [ ]:
historical_rows = fetch_rows(
    """
    SELECT token_id, price_time_utc, price, interval, fidelity
    FROM price_history
    WHERE token_id IN ({placeholders})
    ORDER BY token_id ASC, price_time_utc ASC
    """.format(placeholders=", ".join(["?"] * len(TOKEN_IDS))),
    TOKEN_IDS,
)

if not historical_rows:
    print("No price_history rows were written for the selected market.")
else:
    print(f"Historical price rows: {len(historical_rows)}")
    fig, ax = plt.subplots(figsize=(12, 5))
    for token_id in TOKEN_IDS:
        rows = [row for row in historical_rows if str(row["token_id"]) == token_id]
        valid_rows = [
            (parse_optional_datetime(row["price_time_utc"]), float(row["price"]))
            for row in rows
            if row["price"] is not None
        ]
        if not valid_rows:
            continue
        label = token_label(token_id, OUTCOME_LABELS, TOKEN_INDEX_LOOKUP)
        ax.plot(
            [item[0] for item in valid_rows],
            [item[1] for item in valid_rows],
            label=f"{label} ({token_id[:10]}...)",
        )
    ax.set_title("Historical one-week price history")
    ax.set_ylabel("Price")
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate()
    plt.show()


## 5. Live Top-Of-Book View

Plot live `order_book_snapshots` for mid-price, spread, and top-of-book sizes.


In [ ]:
snapshot_rows = fetch_rows(
    """
    SELECT asset_id, snapshot_time_utc, mid_price, spread, best_bid_size, best_ask_size, best_bid_price, best_ask_price
    FROM order_book_snapshots
    WHERE asset_id IN ({placeholders})
    ORDER BY snapshot_time_utc ASC, asset_id ASC
    """.format(placeholders=", ".join(["?"] * len(TOKEN_IDS))),
    TOKEN_IDS,
)

if not snapshot_rows:
    print("No order_book_snapshots rows were written for this run.")
else:
    print(f"Order-book snapshot rows: {len(snapshot_rows)}")
    fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=True)
    for token_id in TOKEN_IDS:
        rows = [row for row in snapshot_rows if str(row["asset_id"]) == token_id]
        if not rows:
            continue
        label = token_label(token_id, OUTCOME_LABELS, TOKEN_INDEX_LOOKUP)
        times = [parse_optional_datetime(row["snapshot_time_utc"]) for row in rows]
        mid_prices = [float(row["mid_price"]) if row["mid_price"] is not None else None for row in rows]
        spreads = [float(row["spread"]) if row["spread"] is not None else None for row in rows]
        bid_sizes = [float(row["best_bid_size"]) if row["best_bid_size"] is not None else 0.0 for row in rows]
        ask_sizes = [float(row["best_ask_size"]) if row["best_ask_size"] is not None else 0.0 for row in rows]

        mid_points = [(time, value) for time, value in zip(times, mid_prices) if value is not None]
        spread_points = [(time, value) for time, value in zip(times, spreads) if value is not None]
        if mid_points:
            axes[0].plot([item[0] for item in mid_points], [item[1] for item in mid_points], label=label)
        if spread_points:
            axes[1].plot([item[0] for item in spread_points], [item[1] for item in spread_points], label=label)
        axes[2].plot(times, bid_sizes, label=f"{label} bid")
        axes[2].plot(times, ask_sizes, linestyle="--", label=f"{label} ask")

    axes[0].set_title("Live mid-price by token")
    axes[0].set_ylabel("Mid price")
    axes[1].set_title("Live spread by token")
    axes[1].set_ylabel("Spread")
    axes[2].set_title("Best bid/ask size by token")
    axes[2].set_ylabel("Size")
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    for axis in axes:
        axis.legend(loc="best")
    fig.autofmt_xdate()
    plt.show()


## 6. Live Trade View

Summarize `trades`, plot buy and sell markers over the live mid-price series when possible, and chart notional by minute. Empty-trade sessions should still complete cleanly.


In [ ]:
trade_rows = fetch_rows(
    """
    SELECT asset_id, trade_time_utc, side, price, size, usdc_size, proxy_wallet, transaction_hash
    FROM trades
    WHERE asset_id IN ({placeholders})
    ORDER BY trade_time_utc ASC, asset_id ASC
    """.format(placeholders=", ".join(["?"] * len(TOKEN_IDS))),
    TOKEN_IDS,
)

if not trade_rows:
    print("No live trades were captured during this session. The recorder and charts still completed successfully.")
else:
    print(f"Trade rows: {len(trade_rows)}")
    trade_summary = Counter((str(row["asset_id"]), str(row["side"] or "UNKNOWN").upper()) for row in trade_rows)
    pprint(dict(trade_summary))

    notional_by_minute: dict[tuple[str, datetime], float] = defaultdict(float)
    for row in trade_rows:
        trade_time = parse_optional_datetime(row["trade_time_utc"])
        if trade_time is None:
            continue
        minute_bucket = trade_time.replace(second=0, microsecond=0)
        if row["usdc_size"] is not None:
            notional = Decimal(str(row["usdc_size"]))
        elif row["price"] is not None and row["size"] is not None:
            notional = Decimal(str(row["price"])) * Decimal(str(row["size"]))
        else:
            notional = Decimal("0")
        notional_by_minute[(str(row["asset_id"]), minute_bucket)] += float(notional)

    fig, axes = plt.subplots(2, 1, figsize=(13, 10), sharex=False)
    snapshot_lookup: dict[str, list[dict[str, object]]] = defaultdict(list)
    for row in snapshot_rows:
        snapshot_lookup[str(row["asset_id"])] += [row]

    for token_id in TOKEN_IDS:
        label = token_label(token_id, OUTCOME_LABELS, TOKEN_INDEX_LOOKUP)
        token_trades = [row for row in trade_rows if str(row["asset_id"]) == token_id]
        token_snapshots = snapshot_lookup.get(token_id, [])

        mid_points = [
            (parse_optional_datetime(row["snapshot_time_utc"]), float(row["mid_price"]))
            for row in token_snapshots
            if row["mid_price"] is not None
        ]
        if mid_points:
            axes[0].plot([item[0] for item in mid_points], [item[1] for item in mid_points], alpha=0.45, label=f"{label} mid")

        buy_points = [
            (parse_optional_datetime(row["trade_time_utc"]), float(row["price"]))
            for row in token_trades
            if str(row["side"] or "").upper() == "BUY" and row["price"] is not None
        ]
        sell_points = [
            (parse_optional_datetime(row["trade_time_utc"]), float(row["price"]))
            for row in token_trades
            if str(row["side"] or "").upper() == "SELL" and row["price"] is not None
        ]
        if buy_points:
            axes[0].scatter([item[0] for item in buy_points], [item[1] for item in buy_points], marker="^", s=55, label=f"{label} buys")
        if sell_points:
            axes[0].scatter([item[0] for item in sell_points], [item[1] for item in sell_points], marker="v", s=55, label=f"{label} sells")

        minute_points = sorted(
            (minute_bucket, notional)
            for (asset_id, minute_bucket), notional in notional_by_minute.items()
            if asset_id == token_id
        )
        if minute_points:
            axes[1].bar(
                [item[0] for item in minute_points],
                [item[1] for item in minute_points],
                width=1 / 1440,
                alpha=0.6,
                label=f"{label} notional/min",
            )

    axes[0].set_title("Live trades over mid-price")
    axes[0].set_ylabel("Price")
    axes[1].set_title("Live trade notional by minute")
    axes[1].set_ylabel("USDC notional")
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    for axis in axes:
        axis.legend(loc="best")
    fig.autofmt_xdate()
    plt.show()
